# STIR-Net V1 — prediction debugging with Napari

This notebook starts from the trained checkpoint produced by
`05_stirnet_first_overfit.ipynb`.

The first objective is purely diagnostic:

1. load the **latest saved same-sample checkpoint**;
2. rebuild the exact all-cell BlastoSPIM sample;
3. run STIR-Net in evaluation mode;
4. convert the query outputs into the final instance volume using the current V1 inference semantics;
5. compare the prediction against the correct GT volume in **Napari 3D**.

No model patching or training occurs here.

The prediction renderer below is deliberately memory-bounded. It is equivalent to the current V1 postprocessing semantics, but renders one query in spatial chunks instead of materializing all selected full-resolution query masks on the GPU simultaneously.


In [ ]:
from pathlib import Path
import gc
import math
import re
import time

import numpy as np
import pandas as pd
import torch
from scipy import ndimage as ndi

from learned.stirnet import StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config,
    _repo_root,
    build_real_batch,
)
from learned.stirnet.model.query_builder import (
    QUERY_PRIMARY,
    QUERY_SPLIT,
    QUERY_TEMPORAL,
    QUERY_DISCOVERY,
)
from learned.stirnet.training.checkpoint import load_checkpoint
from learned.stirnet.training.trainer import (
    model_forward_from_batch,
    move_to_device,
)

REPO_ROOT = _repo_root(Path.cwd())

DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)

RUN_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "05_same_sample"
)

DEBUG_DIR = RUN_DIR / "debug_06"
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

AMP_DTYPE = torch.float16

# Each query is rendered in bounded GPU chunks.
# The expensive slowdown seen previously was CPU connected-component
# selection, not this renderer.
RENDER_CHUNK_VOXELS = 1_048_576

if not torch.cuda.is_available():
    raise RuntimeError("This prediction-debug notebook requires CUDA.")

device = torch.device("cuda")

print("Repository :", REPO_ROOT)
print("Data       :", DATA_DIR)
print("Run dir    :", RUN_DIR)
print("Debug dir  :", DEBUG_DIR)
print("GPU        :", torch.cuda.get_device_name(0))


## 1. Locate the latest trained checkpoint

In [ ]:
checkpoint_pattern = re.compile(r"checkpoint_step_(\d+)\.pt$")

checkpoint_candidates = []

for path in RUN_DIR.glob("checkpoint_step_*.pt"):
    match = checkpoint_pattern.fullmatch(path.name)
    if match is not None:
        checkpoint_candidates.append(
            (int(match.group(1)), path)
        )

if not checkpoint_candidates:
    raise FileNotFoundError(
        "No resumable checkpoint_step_XXX.pt was found in "
        f"{RUN_DIR}. Run the continuation block in Notebook 05 first."
    )

checkpoint_step, CHECKPOINT_PATH = max(
    checkpoint_candidates,
    key=lambda item: item[0],
)

print("Checkpoint:", CHECKPOINT_PATH)
print("Step      :", checkpoint_step)


## 2. Rebuild the exact all-cell sample

This uses the same source helper as the forward/backward/overfit acceptance experiments. The GT target is represented by one integer label map rather than a dense `K × volume` mask stack.


In [ ]:
prepare_started = time.perf_counter()

batch, sample = build_real_batch(DATA_DIR)

print("ROI shape          :", sample["roi_shape"])
print("Current cells      :", sample["current_count"])
print("GT cells           :", sample["target_count"])
print("Graph nodes        :", sample["graph_nodes"])
print("Temporal tracklets :", sample["temporal_tracklets"])
print("Required queries   :", sample["required_queries"])
print("Preparation time   :", f"{time.perf_counter() - prepare_started:.2f} s")

assert sample["current_count"] == 36
assert sample["target_count"] == 33
assert sample["temporal_tracklets"] == 52
assert sample["required_queries"] == 132

# Keep aligned CPU volumes for visualization before moving model inputs.
raw_norm = batch["spatial_inputs"][0, 0].numpy().astype(np.float32, copy=False)
current_labels = batch["instance_labels"][0].numpy().astype(np.int32, copy=False)
gt_labels = np.asarray(
    batch["targets"][0]["label_map"],
    dtype=np.int32,
)

spacing_zyx_um = (
    batch["spacing_um"][0]
    .numpy()
    .astype(np.float32)
)

print("Raw          :", raw_norm.shape, raw_norm.dtype)
print("Current      :", current_labels.shape, current_labels.dtype)
print("Ground truth :", gt_labels.shape, gt_labels.dtype)
print("Spacing um   :", tuple(float(v) for v in spacing_zyx_um))


## 3. Recreate the reduced-width model and load the trained state

In [ ]:
cfg = _reduced_config()

model = StirNet(cfg).to(device)

checkpoint = load_checkpoint(
    CHECKPOINT_PATH,
    model,
    map_location="cpu",
    strict=True,
)

model.eval()

print("Loaded checkpoint step:", checkpoint.get("step"))
print("Model checkpoint load  : OK")
print("Inference thresholds:")
print("  render exist :", cfg.inference.render_exist_threshold)
print("  final exist  :", cfg.inference.final_exist_threshold)
print("  mask         :", cfg.inference.mask_threshold)
print("  min voxels   :", cfg.inference.min_mask_voxels)


## 4. Move only model inputs to CUDA

Large target maps remain on CPU. The network receives the same FP16 spatial input used by the validated acceptance gates.


In [ ]:
batch_device = {}

for key, value in batch.items():
    if key == "targets":
        batch_device[key] = value
    elif key == "spatial_inputs":
        batch_device[key] = value.to(
            device=device,
            dtype=AMP_DTYPE,
        )
    elif key == "instance_labels":
        batch_device[key] = value.to(
            device=device,
            dtype=torch.int32,
        )
    else:
        batch_device[key] = move_to_device(
            value,
            device,
        )

del batch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(
    "CUDA allocated before forward:",
    f"{torch.cuda.memory_allocated() / 1024**3:.3f} GiB",
)


## 5. Run the trained model once in evaluation mode

In [ ]:
forward_started = time.perf_counter()

with torch.no_grad(), torch.autocast(
    device_type="cuda",
    dtype=AMP_DTYPE,
):
    outputs = model_forward_from_batch(
        model,
        batch_device,
    )

torch.cuda.synchronize()

forward_seconds = time.perf_counter() - forward_started
forward_peak_gib = torch.cuda.max_memory_allocated() / 1024**3

for name, tensor in {
    "exist_logits": outputs.exist_logits,
    "centers_cellscale": outputs.centers_cellscale,
    "coarse_mask_logits": outputs.coarse_mask_logits,
    "native_mask_embeddings": outputs.native_mask_embeddings,
    "mask_features": outputs.mask_features,
}.items():
    finite = bool(torch.isfinite(tensor.float()).all())
    print(
        f"{name:24s} "
        f"shape={tuple(tensor.shape)} "
        f"dtype={tensor.dtype} "
        f"finite={finite}"
    )
    if not finite:
        raise RuntimeError(f"{name} is non-finite.")

print()
print("Forward time     :", f"{forward_seconds:.2f} s")
print("Forward peak CUDA:", f"{forward_peak_gib:.3f} GiB")


## 6. Inspect query existence predictions before rendering masks

The final postprocessor first uses the existence head to decide which query masks can become output instances.

This table is useful because a mask can only appear in the final volume if its query survives the final existence threshold.


In [ ]:
exist_probs = (
    torch.sigmoid(outputs.exist_logits[0])
    .masked_fill(outputs.query_padding_mask[0], 0)
    .detach()
    .float()
    .cpu()
    .numpy()
)

query_types = (
    outputs.query_types[0]
    .detach()
    .cpu()
    .numpy()
)

source_ids = (
    outputs.source_instance_ids[0]
    .detach()
    .cpu()
    .numpy()
)

type_names = {
    int(QUERY_PRIMARY): "primary",
    int(QUERY_SPLIT): "split",
    int(QUERY_TEMPORAL): "temporal",
    int(QUERY_DISCOVERY): "discovery",
}

query_table = pd.DataFrame(
    {
        "query": np.arange(len(exist_probs)),
        "exist_prob": exist_probs,
        "type": [
            type_names.get(int(qt), f"type_{int(qt)}")
            for qt in query_types
        ],
        "source_instance_id": source_ids,
    }
)

query_table = query_table.sort_values(
    "exist_prob",
    ascending=False,
).reset_index(drop=True)

render_candidates = int(
    np.count_nonzero(
        exist_probs
        > cfg.inference.render_exist_threshold
    )
)

final_candidates = int(
    np.count_nonzero(
        exist_probs
        >= cfg.inference.final_exist_threshold
    )
)

print("Total queries           :", len(exist_probs))
print("Render-threshold queries:", render_candidates)
print("Final-existence queries :", final_candidates)
print("Exist probability sum   :", float(exist_probs.sum()))
print()
display(query_table.head(30))


## 7. Memory-bounded native mask renderer

The repository's native mask renderer computes:

- query-mask embedding × full-resolution mask features;
- the seeded current-instance prior for primary/split queries;
- the physical Gaussian prior for temporal queries.

For this large ROI, rendering all surviving queries as one dense `[Q,Z,Y,X]` tensor is unnecessary for visualization. The helper below computes exactly the same per-query logits in bounded voxel chunks, then transfers one probability volume at a time to CPU.


In [ ]:
def render_one_query_probability(
    outputs,
    query_index: int,
    *,
    chunk_voxels: int = RENDER_CHUNK_VOXELS,
) -> np.ndarray:

    b = 0

    _, channels, z_size, y_size, x_size = outputs.mask_features.shape
    shape = (z_size, y_size, x_size)
    voxel_count = z_size * y_size * x_size

    feature_flat = (
        outputs.mask_features[b]
        .reshape(channels, -1)
    )

    embedding = outputs.native_mask_embeddings[b, query_index]

    current_flat = (
        outputs.instance_labels[b]
        .reshape(-1)
    )

    query_type = int(
        outputs.query_types[b, query_index].item()
    )

    source_id = int(
        outputs.source_instance_ids[b, query_index].item()
    )

    spacing = outputs.spacing_um[b].float()
    dref = outputs.dref_um[b].float()

    center_rel_um = (
        outputs.centers_cellscale[b, query_index].float()
        * dref
    )

    extent_um = (
        torch.tensor(
            [
                z_size - 1,
                y_size - 1,
                x_size - 1,
            ],
            device=device,
            dtype=torch.float32,
        )
        * spacing
    )

    probability = np.empty(
        voxel_count,
        dtype=np.float32,
    )

    for start in range(
        0,
        voxel_count,
        chunk_voxels,
    ):
        end = min(
            start + chunk_voxels,
            voxel_count,
        )

        with torch.no_grad(), torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        ):
            logits = torch.einsum(
                "c,cv->v",
                embedding,
                feature_flat[:, start:end],
            )

        # Same seeded prior used by render_native_masks().
        if (
            query_type in (
                int(QUERY_PRIMARY),
                int(QUERY_SPLIT),
            )
            and source_id >= 0
        ):
            inside = (
                current_flat[start:end]
                == source_id
            )

            prior = torch.where(
                inside,
                logits.new_tensor(
                    cfg.queries.prior_inside_logit
                ),
                logits.new_tensor(
                    cfg.queries.prior_outside_logit
                ),
            )

            logits = logits + prior

        # Same physical Gaussian temporal prior used by
        # render_native_masks().
        elif query_type == int(QUERY_TEMPORAL):

            linear = torch.arange(
                start,
                end,
                device=device,
                dtype=torch.long,
            )

            z_coord = torch.div(
                linear,
                y_size * x_size,
                rounding_mode="floor",
            )

            remainder = linear.remainder(
                y_size * x_size
            )

            y_coord = torch.div(
                remainder,
                x_size,
                rounding_mode="floor",
            )

            x_coord = remainder.remainder(
                x_size
            )

            coords = torch.stack(
                [
                    z_coord,
                    y_coord,
                    x_coord,
                ],
                dim=-1,
            ).float()

            coords_um = (
                coords * spacing[None]
                - 0.5 * extent_um[None]
            )

            delta = (
                coords_um
                - center_rel_um[None]
            )

            dist2 = (
                delta.square()
                .sum(dim=-1)
            )

            sigma = (
                cfg.queries.temporal_gaussian_sigma_dref
                * dref
            )

            logits = (
                logits
                + cfg.queries.prior_inside_logit
                * torch.exp(
                    -0.5
                    * dist2
                    / sigma.clamp_min(1e-6).square()
                ).to(logits.dtype)
            )

        probability[start:end] = (
            torch.sigmoid(logits.float())
            .cpu()
            .numpy()
        )

        del logits

    return probability.reshape(shape)


print(
    "Renderer ready; chunk voxels =",
    RENDER_CHUNK_VOXELS,
)


## 8. Build the final STIR-Net instance volume

This section now has two cells: first the fast **exact** connected-component selector, then the complete prediction loop. Run both cells in order. The prediction loop always initializes `prediction_labels`, `best_score`, `accepted_records`, and `accepted_df` from scratch, so interrupted older runs cannot leak stale state into the diagnostics.

The final semantics remain: existence filtering, mask thresholding, nearest-center connected component, minimum mask size, and voxel-wise `existence × mask_probability` winner assignment.


In [ ]:
def component_near_center(
    mask: np.ndarray,
    center_vox: np.ndarray,
    min_voxels: int,
) -> np.ndarray:
    """Exact source-equivalent component selection without repeated full scans.

    The old diagnostic implementation called np.argwhere(cc == label) for every
    connected component, repeatedly scanning the entire 3D ROI. Discovery
    queries can create many components, making that path extremely slow.

    This version:
      1. labels once;
      2. computes all component sizes in one bincount;
      3. uses each component's bounding box;
      4. computes the exact nearest-voxel distance only inside that bbox.

    Selection semantics remain: keep the component containing the predicted
    center when valid, otherwise keep the valid component nearest the center.
    """

    cc, count = ndi.label(mask)

    if count == 0:
        return np.zeros_like(mask, dtype=bool)

    center = np.rint(center_vox).astype(np.int64)
    shape_arr = np.asarray(mask.shape, dtype=np.int64)

    # Component sizes for every label in one pass.
    sizes = np.bincount(
        cc.ravel(),
        minlength=count + 1,
    )

    # Fast path: center lands inside a sufficiently large component.
    if (
        np.all(center >= 0)
        and np.all(center < shape_arr)
    ):
        center_label = int(cc[tuple(center)])

        if (
            center_label > 0
            and sizes[center_label] >= min_voxels
        ):
            return cc == center_label

    valid_labels = np.flatnonzero(
        sizes >= min_voxels
    )
    valid_labels = valid_labels[
        valid_labels != 0
    ]

    if valid_labels.size == 0:
        return np.zeros_like(mask, dtype=bool)

    # One bounding box per component. Unlike the old code, we never rescan
    # the full 27.5M-voxel ROI separately for every label.
    objects = ndi.find_objects(cc)

    best_label = None
    best_distance_sq = np.inf

    for label in valid_labels:
        label = int(label)
        slc = objects[label - 1]

        if slc is None:
            continue

        local_component = (
            cc[slc] == label
        )

        local_points = np.argwhere(
            local_component
        )

        if local_points.size == 0:
            continue

        offset = np.asarray(
            [axis.start for axis in slc],
            dtype=np.int64,
        )

        points = (
            local_points
            + offset[None]
        )

        delta = (
            points
            - center[None]
        )

        distance_sq = float(
            np.min(
                np.sum(
                    delta * delta,
                    axis=1,
                )
            )
        )

        if distance_sq < best_distance_sq:
            best_distance_sq = distance_sq
            best_label = label

    if best_label is None:
        return np.zeros_like(mask, dtype=bool)

    return cc == best_label


print("Fast exact component selector ready.")


In [ ]:
# Rebuild the prediction from scratch.
#
# This cell intentionally initializes every result variable locally so it
# cannot accidentally reuse a partial prediction from an interrupted run.

shape = tuple(
    int(v)
    for v in outputs.instance_labels.shape[-3:]
)

extent_um = (
    np.asarray(shape, dtype=np.float32)
    - 1
) * spacing_zyx_um

prediction_labels = np.zeros(
    shape,
    dtype=np.int32,
)

best_score = np.full(
    shape,
    -np.inf,
    dtype=np.float32,
)

accepted_columns = [
    "output_label",
    "query",
    "exist_prob",
    "type",
    "source_instance_id",
    "mask_voxels_before_competition",
    "mask_voxels_final",
    "pred_center_z",
    "pred_center_y",
    "pred_center_x",
    "render_seconds",
    "component_seconds",
    "competition_seconds",
]

accepted_records = []

# postprocess_batch first renders > render threshold and then rejects
# existence < final threshold. Since final threshold is higher here,
# these are exactly the queries that can contribute to the final result.
candidate_indices = np.flatnonzero(
    exist_probs
    >= cfg.inference.final_exist_threshold
)

print(
    "Queries to render after final existence threshold:",
    len(candidate_indices),
)

render_all_started = time.perf_counter()

for candidate_number, query_index in enumerate(
    candidate_indices,
    start=1,
):
    query_index = int(query_index)

    exist_probability = float(
        exist_probs[query_index]
    )

    query_type_name = type_names.get(
        int(query_types[query_index]),
        f"type_{int(query_types[query_index])}",
    )

    print(
        f"[{candidate_number:02d}/{len(candidate_indices):02d}] "
        f"query={query_index:3d} "
        f"type={query_type_name:9s} "
        f"exist={exist_probability:.4f}",
        flush=True,
    )

    # --------------------------------------------------------------
    # Native mask probability
    # --------------------------------------------------------------
    query_started = time.perf_counter()

    mask_probability = render_one_query_probability(
        outputs,
        query_index,
    )

    render_seconds = (
        time.perf_counter()
        - query_started
    )

    # --------------------------------------------------------------
    # Predicted center in voxel coordinates
    # --------------------------------------------------------------
    center_rel_um = (
        outputs.centers_cellscale[0, query_index]
        .detach()
        .float()
        .cpu()
        .numpy()
        * float(outputs.dref_um[0].item())
    )

    center_vox = (
        center_rel_um
        + 0.5 * extent_um
    ) / spacing_zyx_um

    # --------------------------------------------------------------
    # Threshold + exact nearest connected component
    # --------------------------------------------------------------
    component_started = time.perf_counter()

    binary_mask = (
        mask_probability
        > cfg.inference.mask_threshold
    )

    binary_mask = component_near_center(
        binary_mask,
        center_vox,
        cfg.inference.min_mask_voxels,
    )

    component_seconds = (
        time.perf_counter()
        - component_started
    )

    mask_voxels = int(
        np.count_nonzero(binary_mask)
    )

    if mask_voxels < cfg.inference.min_mask_voxels:
        print(
            f"    rejected: voxels={mask_voxels} "
            f"render={render_seconds:.2f}s "
            f"component={component_seconds:.2f}s"
        )

        del mask_probability, binary_mask
        gc.collect()
        continue

    # --------------------------------------------------------------
    # Streaming winner assignment.
    #
    # Equivalent to source postprocessing's stack + argmax, but without
    # materializing [Naccepted, Z, Y, X] scores in CPU RAM.
    # --------------------------------------------------------------
    competition_started = time.perf_counter()

    score = (
        exist_probability
        * mask_probability
    )

    update = (
        binary_mask
        & (score > best_score)
    )

    best_score[update] = score[update]

    output_label = len(accepted_records) + 1

    prediction_labels[update] = output_label

    competition_seconds = (
        time.perf_counter()
        - competition_started
    )

    accepted_records.append(
        {
            "output_label": output_label,
            "query": query_index,
            "exist_prob": exist_probability,
            "type": query_type_name,
            "source_instance_id": int(
                source_ids[query_index]
            ),
            "mask_voxels_before_competition": mask_voxels,
            "mask_voxels_final": 0,  # filled after all competitions
            "pred_center_z": float(center_vox[0]),
            "pred_center_y": float(center_vox[1]),
            "pred_center_x": float(center_vox[2]),
            "render_seconds": float(render_seconds),
            "component_seconds": float(component_seconds),
            "competition_seconds": float(competition_seconds),
        }
    )

    print(
        f"    accepted label={output_label:2d} "
        f"voxels={mask_voxels:8d} "
        f"render={render_seconds:.2f}s "
        f"component={component_seconds:.2f}s "
        f"competition={competition_seconds:.2f}s"
    )

    del (
        mask_probability,
        binary_mask,
        score,
        update,
    )

    gc.collect()


render_seconds_total = (
    time.perf_counter()
    - render_all_started
)

# Always define the dataframe, including for zero accepted queries.
accepted_df = pd.DataFrame(
    accepted_records,
    columns=accepted_columns,
)

# Winner competition can take voxels away from earlier masks.
for row_index, output_label in enumerate(
    accepted_df.get(
        "output_label",
        pd.Series(dtype=int),
    )
):
    accepted_df.loc[
        row_index,
        "mask_voxels_final",
    ] = int(
        np.count_nonzero(
            prediction_labels == int(output_label)
        )
    )

print()
print("Accepted output instances:", len(accepted_df))
print("Total rendering/postproc   :", f"{render_seconds_total:.2f} s")
print(
    "Prediction foreground voxels:",
    int(np.count_nonzero(prediction_labels)),
)

if len(accepted_df):
    display(accepted_df)
else:
    print("No queries survived final mask postprocessing.")


## 9. Quantify the foreground collapse/coverage before opening Napari

This is **not** the competition metric and does not judge instance identity. It only compares predicted foreground occupancy with GT foreground occupancy.

For the current debugging question it is useful because the 25-step loss history suggested that masks may have become very small.


In [ ]:
gt_foreground = gt_labels > 0
pred_foreground = prediction_labels > 0
current_foreground = current_labels > 0

intersection = int(
    np.count_nonzero(
        gt_foreground
        & pred_foreground
    )
)

gt_voxels = int(
    np.count_nonzero(
        gt_foreground
    )
)

pred_voxels = int(
    np.count_nonzero(
        pred_foreground
    )
)

current_voxels = int(
    np.count_nonzero(
        current_foreground
    )
)

foreground_dice = (
    2.0 * intersection
    / max(
        gt_voxels + pred_voxels,
        1,
    )
)

foreground_precision = (
    intersection
    / max(pred_voxels, 1)
)

foreground_recall = (
    intersection
    / max(gt_voxels, 1)
)

print("Current foreground voxels :", current_voxels)
print("GT foreground voxels      :", gt_voxels)
print("Pred foreground voxels    :", pred_voxels)
print(
    "Pred / GT foreground ratio:",
    f"{pred_voxels / max(gt_voxels, 1):.4f}",
)
print("Foreground precision      :", f"{foreground_precision:.4f}")
print("Foreground recall         :", f"{foreground_recall:.4f}")
print("Foreground Dice           :", f"{foreground_dice:.4f}")


## 10. Build direct comparison volumes and save diagnostic arrays

The comparison layer uses:

- `1` = GT foreground only — the model **missed** these voxels;
- `2` = prediction foreground only — the model **added** these voxels;
- `3` = GT/prediction overlap.

The saved arrays let us reopen the visualization without rerunning the neural network.


In [ ]:
comparison = np.zeros(
    shape,
    dtype=np.uint8,
)

gt_only = (
    gt_foreground
    & ~pred_foreground
)

pred_only = (
    pred_foreground
    & ~gt_foreground
)

overlap = (
    gt_foreground
    & pred_foreground
)

comparison[gt_only] = 1
comparison[pred_only] = 2
comparison[overlap] = 3

prediction_path = (
    DEBUG_DIR
    / f"prediction_labels_step_{checkpoint_step:03d}.npy"
)

comparison_path = (
    DEBUG_DIR
    / f"foreground_comparison_step_{checkpoint_step:03d}.npy"
)

accepted_path = (
    DEBUG_DIR
    / f"accepted_queries_step_{checkpoint_step:03d}.csv"
)

np.save(
    prediction_path,
    prediction_labels,
)

np.save(
    comparison_path,
    comparison,
)

accepted_df.to_csv(
    accepted_path,
    index=False,
)

print("Saved prediction :", prediction_path)
print("Saved comparison :", comparison_path)
print("Saved query table:", accepted_path)

# Napari uses the GPU through OpenGL. The neural-network outputs are no
# longer needed, so release PyTorch CUDA state before opening the viewer.
del outputs
del model
del batch_device

gc.collect()
torch.cuda.empty_cache()

print(
    "PyTorch CUDA allocated before Napari:",
    f"{torch.cuda.memory_allocated() / 1024**3:.3f} GiB",
)


## 11. Napari 3D visualization

The viewer opens with:

- **Raw normalized** — the actual image input;
- **STIR-Net prediction** — final postprocessed instance labels;
- **Ground truth** — correct instance labels;
- **Current initialization** — the imperfect connected-component segmentation fed to STIR-Net;
- **Foreground comparison** — direct GT-only / prediction-only / overlap map.

The comparison layer is initially visible because it most directly reveals whether the trained network collapsed its masks.

Toggle the individual label layers in Napari to inspect cells one by one.


In [ ]:
try:
    import napari
except ImportError as exc:
    raise ImportError(
        "Napari is not installed in the active project environment."
    ) from exc

# Integrate Qt with the Jupyter kernel.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("gui", "qt")

viewer = napari.Viewer(
    ndisplay=3,
    title=(
        f"STIR-Net V1 prediction debug — "
        f"checkpoint step {checkpoint_step}"
    ),
)

viewer.add_image(
    raw_norm,
    name="Raw normalized",
    scale=tuple(spacing_zyx_um),
    colormap="gray",
    rendering="mip",
    opacity=0.65,
)

viewer.add_labels(
    current_labels,
    name="Current initialization",
    scale=tuple(spacing_zyx_um),
    opacity=0.45,
    visible=False,
)

viewer.add_labels(
    gt_labels,
    name="Ground truth",
    scale=tuple(spacing_zyx_um),
    opacity=0.55,
    visible=False,
)

viewer.add_labels(
    prediction_labels,
    name=f"STIR-Net prediction step {checkpoint_step}",
    scale=tuple(spacing_zyx_um),
    opacity=0.65,
    visible=False,
)

comparison_name = (
    "Foreground comparison: "
    "1=GT only, 2=Pred only, 3=Overlap"
)

# Napari's Labels color argument has changed across releases.
# Prefer the semantic colors, but fall back to the default label palette
# instead of failing the entire visualization.
try:
    viewer.add_labels(
        comparison,
        name=comparison_name,
        scale=tuple(spacing_zyx_um),
        opacity=0.75,
        visible=True,
        color={
            1: "magenta",
            2: "cyan",
            3: "lime",
        },
    )
except (TypeError, ValueError):
    viewer.add_labels(
        comparison,
        name=comparison_name,
        scale=tuple(spacing_zyx_um),
        opacity=0.75,
        visible=True,
    )

# Add accepted predicted centers as points.
if len(accepted_df):
    predicted_centers_zyx = accepted_df[
        [
            "pred_center_z",
            "pred_center_y",
            "pred_center_x",
        ]
    ].to_numpy(
        dtype=np.float32
    )

    point_kwargs = dict(
        name="Predicted centers",
        scale=tuple(spacing_zyx_um),
        size=5,
        face_color="yellow",
        visible=False,
    )

    # edge_color was renamed in newer Napari releases.
    try:
        viewer.add_points(
            predicted_centers_zyx,
            edge_color="black",
            **point_kwargs,
        )
    except TypeError:
        viewer.add_points(
            predicted_centers_zyx,
            border_color="black",
            **point_kwargs,
        )

print("Napari viewer opened.")
print()
print("Suggested first inspection:")
print("  1. Raw normalized + Foreground comparison")
print("  2. Toggle Ground truth ON")
print("  3. Toggle STIR-Net prediction ON/OFF")
print("  4. Toggle Current initialization for reference")
print("  5. Inspect predicted centers if masks are tiny or absent")


## Stop here first

Before adding more diagnostics, inspect the 3D result carefully.

The next debugging stage should depend on what Napari shows:

- **tiny/empty predicted masks** → inspect native mask probabilities inside/outside matched GT cells;
- **reasonable masks in wrong locations** → inspect center/query matching;
- **correct coarse structure but bad native masks** → compare coarse masks against native rendered masks;
- **many missing instances** → inspect existence probabilities and query types;
- **reasonable masks but wrong ownership where cells touch** → inspect overlap/winner assignment.

Do not train further until we know which failure mode is actually present.
